In [1]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False
import sys

print(sys.executable)  #
from pprint import pprint
import polars as pl
from pathlib import Path

/home/nasim/Desktop/work/rag/sec-10k-rag/.venv/bin/python


In [2]:
from groq import Groq

In [3]:
from sec_10k_rag.common.common import write_to_parquet_file, load_parquet

In [4]:
from sec_10k_rag.step_07_search_quality.config import get_config, ConfigType, get_llm_api_key, ROOT_DIR

In [5]:
from sec_10k_rag.step_07_search_quality.lib import generate_questions, load_chunks_text_from_db, sample_chunks, get_question_vectors, find_nearest_neighbors

In [ ]:
print(dir(ConfigType)) 

In [6]:
config = get_config(ConfigType.GENERATE_QUESTIONS)

In [ ]:
print(config)

In [7]:
instructions = config.instructions

In [8]:
chunks_df = load_chunks_text_from_db(instructions.db, instructions.chunks_table)

In [ ]:
display(chunks_df.head(10))

In [9]:
sample_df = sample_chunks(chunks_df, n=instructions.sample_size, seed=instructions.seed)

In [ ]:
print(sample_df.height)

In [ ]:
print(chunks_df.height)

In [ ]:
pprint(sorted(m.id for m in Groq(api_key=get_llm_api_key()).models.list().data))

In [10]:
questions = generate_questions(
    sample=sample_df,
    llm_info=instructions.llm_info,
    company_names=instructions.company_names
)

TypeError: APIStatusError.__init__() missing 2 required keyword-only arguments: 'response' and 'body'

In [ ]:
display(questions)

In [11]:
questions_path = (
    Path(config.processed_data_dir) / instructions.output_file_name
)

In [ ]:
print(questions_path)

In [ ]:
# write_to_parquet_file(questions, questions_path)

In [ ]:
loaded_questions_df = load_parquet(questions_path)

In [ ]:
print(loaded_questions_df.height)

In [12]:
questions_vector_df = get_question_vectors(questions_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
display(questions_vector_df.head(10))

question_id,section_id,chunk_index,ticker,fiscal_year,item,question,answer,quote,query,embedding
u32,str,i64,str,i64,str,str,str,str,str,"array[f32, 384]"
0,"""MSFT-2024-item7""",4,"""MSFT""",2024,"""7""","""What potential supply-chain ri…","""Extended disruptions at these …","""Extended disruptions at these …","""Represent this sentence for se…","[0.004262, -0.013034, … 0.042341]"
1,"""NVDA-2024-item1A""",6,"""NVDA""",2024,"""1A""","""What cloud service did Nvidia …","""NVIDIA DGX Cloud services""","""We have begun offering enterpr…","""Represent this sentence for se…","[0.003905, -0.033177, … 0.062534]"
2,"""TSLA-2023-item7""",1,"""TSLA""",2023,"""7""","""How many consumer vehicles did…","""Tesla produced 1,845,985 consu…","""In 2023, we produced 1,845,985…","""Represent this sentence for se…","[0.058794, -0.008197, … 0.040833]"
3,"""MSFT-2024-item1A""",7,"""MSFT""",2024,"""1A""","""What risks does Microsoft cite…","""Microsoft says its focus on cl…","""These costs will reduce the op…","""Represent this sentence for se…","[-0.006373, -0.089215, … 0.021918]"
4,"""NVDA-2024-item15""",45,"""NVDA""",2024,"""15""","""What was Nvidia's weighted ave…","""6.1 years""","""weighted average remaining lea…","""Represent this sentence for se…","[0.010412, -0.016273, … 0.03695]"
5,"""TSLA-2023-item1A""",14,"""TSLA""",2023,"""1A""","""What competitive challenge doe…","""Tesla states that it encounter…","""We face strong competition for…","""Represent this sentence for se…","[-0.029484, 0.013314, … 0.007143]"
6,"""MSFT-2025-item1""",30,"""MSFT""",2025,"""1""","""What supply chain challenge do…","""Microsoft notes that there are…","""There are few qualified suppli…","""Represent this sentence for se…","[-0.070096, -0.026991, … -0.016247]"
7,"""NVDA-2025-item1""",51,"""NVDA""",2025,"""1""","""In Nvidia's 2025 fiscal year, …","""She joined NVIDIA in 2007 as S…","""joined NVIDIA in 2007 as Senio…","""Represent this sentence for se…","[-0.055616, 0.005175, … 0.007479]"
8,"""TSLA-2023-item8""",149,"""TSLA""",2023,"""8""","""What indemnification agreement…","""Tesla entered into an indemnif…","""indemnification agreement with…","""Represent this sentence for se…","[-0.031447, -0.033908, … 0.03926]"


In [ ]:
pprint(config)

In [18]:
neighbors_df = find_nearest_neighbors(questions_vector_df, instructions.db, instructions.chunks_table, top_k=5)

TypeError: find_nearest_neighbors() missing 1 required positional argument: 'filter_columns'

In [ ]:
with pl.Config(tbl_rows=-1): display(neighbors_df)

In [28]:
from sec_10k_rag.step_07_search_quality.lib import find_keyword_matches, fuse_ranks, score, rank_of_source

In [20]:
matches = find_keyword_matches(
    questions_vector_df, db=instructions.db, table="chunks", top_k=30,
    filter_columns=("ticker", "fiscal_year"),
)
matches.group_by("question_id").len().describe()

statistic,question_id,len
str,f64,f64
"""count""",49.0,49.0
"""null_count""",0.0,0.0
"""mean""",24.0,30.0
"""std""",14.28869,0.0
"""min""",0.0,30.0
"""25%""",12.0,30.0
"""50%""",24.0,30.0
"""75%""",36.0,30.0
"""max""",48.0,30.0


In [21]:
vectors = find_nearest_neighbors(
    questions_vector_df, db=instructions.db, table="chunks", top_k=30,
    filter_columns=("ticker", "fiscal_year"),
)


In [23]:
fused = fuse_ranks(vectors, matches, rrf_k=60, top_k=10)

In [24]:
print(fused.height) 

490


In [29]:
print(score(rank_of_source(questions_vector_df, fused), (1, 3, 5, 10)))

shape: (1, 5)
┌──────────┬──────────┬──────────┬───────────┬──────────┐
│ recall@1 ┆ recall@3 ┆ recall@5 ┆ recall@10 ┆ mrr      │
│ ---      ┆ ---      ┆ ---      ┆ ---       ┆ ---      │
│ f64      ┆ f64      ┆ f64      ┆ f64       ┆ f64      │
╞══════════╪══════════╪══════════╪═══════════╪══════════╡
│ 0.44898  ┆ 0.632653 ┆ 0.653061 ┆ 0.77551   ┆ 0.548607 │
└──────────┴──────────┴──────────┴───────────┴──────────┘
